# Classroom Vision · 시연 실행

**Windows / Mac · CUDA / MPS / CPU 자동 선택**

1. VS Code에서 이 노트북을 열고 **로컬 Python 3.13 커널**을 선택합니다.
2. **전체 실행**을 누릅니다. 최초 한 번만 패키지와 보조 모델을 준비합니다.
3. 아래 실행 패널에서 카메라 또는 영상을 고르고 **실행**을 누릅니다.

시연 화면은 별도 창으로 열립니다. 종료는 창의 `Q`·`Esc` 또는 아래 **중지** 버튼입니다.
파인튜닝 가중치는 기본 적용되며 서버 연결은 사용하지 않습니다.

In [ ]:
import importlib.metadata as metadata
from pathlib import Path
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Python 3.13 로컬 커널을 선택한 뒤 다시 실행하세요.")

DEPLOY = next((p for p in (Path.cwd(), Path.cwd() / "deploy")
               if (p / "app.py").is_file() and (p / "models/manifest.json").is_file()), None)
if DEPLOY is None:
    raise RuntimeError("deploy 폴더 안의 START.ipynb를 열어 주세요.")
DEPLOY = DEPLOY.resolve()

from packaging.requirements import Requirement
requirements = [Requirement(line.strip()) for line in (DEPLOY / "requirements.txt").read_text().splitlines()
                if line.strip() and not line.lstrip().startswith("#")]
missing = []
for requirement in requirements:
    try:
        if not requirement.specifier.contains(metadata.version(requirement.name)):
            missing.append(requirement.name)
    except metadata.PackageNotFoundError:
        missing.append(requirement.name)

if missing:
    print("필요한 패키지를 준비합니다. 최초 설치에는 시간이 걸릴 수 있습니다.")
    log_dir = DEPLOY / "outputs"
    log_dir.mkdir(exist_ok=True)
    with (log_dir / "install.log").open("w", encoding="utf-8") as log:
        result = subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(DEPLOY / "requirements.txt")],
                                stdout=log, stderr=subprocess.STDOUT)
    if result.returncode:
        print((log_dir / "install.log").read_text(encoding="utf-8", errors="replace")[-5000:])
        raise RuntimeError("패키지 설치를 확인한 뒤 이 셀을 다시 실행하세요.")
    print("설치 완료. 이미 다른 버전의 torch 등을 불러왔던 커널은 재시작 후 전체 실행하세요.")
else:
    print("필수 패키지 준비 완료")

### 모델 준비

동봉된 파인튜닝 모델과 보조 모델 5개를 확인합니다.
최초 준비 후에는 `models/`를 그대로 보관하면 인터넷 없이 실행할 수 있습니다.

In [ ]:
result = subprocess.run([sys.executable, str(DEPLOY / "models.py"), "--prepare"],
                        cwd=DEPLOY, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout.strip())
if result.returncode:
    raise RuntimeError("모델 준비에 실패했습니다. 위 메시지를 확인한 뒤 다시 실행하세요.")

### 실행 패널

- **자동**은 사용 가능한 CUDA → Mac MPS → CPU 순서로 고릅니다.
- 영상은 파일 경로를 입력합니다. HDR MOV도 자동 변환합니다.
- **환경 확인**은 카메라를 켜지 않고 모델 추론을 확인합니다.

In [ ]:
import html
import os
import platform
import signal
from threading import Thread
import time
import ipywidgets as widgets
from IPython.display import display
import torch

available = [("CPU", "cpu")]
if torch.cuda.is_available():
    available.insert(0, (f"CUDA · {torch.cuda.get_device_name(0)}", "cuda:0"))
if torch.backends.mps.is_available():
    available.insert(0 if not torch.cuda.is_available() else 1, ("Mac MPS", "mps"))
automatic = "cuda:0" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
state = globals().setdefault("_classroom_demo", {"process": None, "log": None})

source = widgets.ToggleButtons(options=[("카메라", "camera"), ("영상", "video")], description="입력")
camera = widgets.BoundedIntText(value=0, min=0, max=20, description="카메라 번호")
sample_video = Path.home() / "Downloads" / "IMG_6463.MOV"
video = widgets.Text(value=str(sample_video) if sample_video.is_file() else "", description="영상 경로",
                     placeholder="예: C:\\Videos\\demo.mp4 또는 /Users/.../demo.MOV",
                     layout=widgets.Layout(width="95%"))
video.layout.display = "none"
device = widgets.Dropdown(options=[(f"자동 · {automatic.upper()}", "auto")] + available,
                          value="auto", description="처리 장치", layout=widgets.Layout(width="95%"))
size = widgets.Dropdown(options=[("정확도 우선 · 960", 960), ("속도 우선 · 640", 640)], description="탐지 설정")
fullscreen = widgets.Checkbox(value=False, description="발표용 전체 화면")
run_button = widgets.Button(description="실행", button_style="success", icon="play")
stop_button = widgets.Button(description="중지", button_style="warning", icon="stop")
check_button = widgets.Button(description="환경 확인", icon="check")
status = widgets.HTML("<b>준비 완료</b> · 실행 버튼을 눌러 주세요.")
output = widgets.Output()


def set_source(change):
    camera.layout.display = "" if change["new"] == "camera" else "none"
    video.layout.display = "" if change["new"] == "video" else "none"


def busy():
    process = state.get("process")
    return process is not None and process.poll() is None


def monitor(process, check, log):
    returncode = process.wait()
    log.close()
    if state.get("process") is not process:
        return
    run_button.disabled = False
    check_button.disabled = False
    status.value = "<b>환경 확인 완료</b>" if check and returncode == 0 else "<b>종료됨</b>"
    if check or returncode not in (0, -15):
        with output:
            print((DEPLOY / "outputs/demo.log").read_text(encoding="utf-8", errors="replace")[-5000:])


def launch(check=False):
    if busy():
        status.value = "이미 실행 중입니다. 기존 창을 닫거나 중지를 눌러 주세요."
        return
    command = [sys.executable, str(DEPLOY / "app.py"), "--offline", "--device", device.value,
               "--size", str(size.value)]
    if check:
        command += ["--check"]
    elif source.value == "video":
        path = Path(video.value.strip().strip('"').strip("'")).expanduser()
        if not video.value.strip() or not path.is_file():
            status.value = "<b>영상 파일 경로를 확인해 주세요.</b>"
            return
        command += ["--video", str(path.resolve())]
    else:
        command += ["--camera", str(camera.value)]
    if fullscreen.value and not check:
        command += ["--fullscreen"]
    (DEPLOY / "outputs").mkdir(exist_ok=True)
    log = (DEPLOY / "outputs/demo.log").open("w", encoding="utf-8")
    options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {"start_new_session": True}
    try:
        process = subprocess.Popen(command, cwd=DEPLOY, stdout=log, stderr=subprocess.STDOUT, **options)
    except OSError as error:
        log.close()
        status.value = html.escape(str(error))
        return
    state.update(process=process, log=log)
    output.clear_output()
    run_button.disabled = True
    check_button.disabled = True
    status.value = "<b>환경을 확인하고 있습니다…</b>" if check else "<b>시연 창 준비 중…</b> · 별도 창이 열립니다."
    Thread(target=monitor, args=(process, check, log), daemon=True).start()


def stop(_):
    process = state.get("process")
    if not busy():
        status.value = "실행 중인 시연이 없습니다."
        return
    status.value = "종료 중…"
    try:
        if os.name == "nt":
            process.send_signal(signal.CTRL_BREAK_EVENT)
        else:
            process.terminate()
    except OSError:
        process.terminate()
    def finish():
        try:
            process.wait(timeout=8)
        except subprocess.TimeoutExpired:
            process.kill()
    Thread(target=finish, daemon=True).start()

source.observe(set_source, names="value")
run_button.on_click(lambda _: launch())
check_button.on_click(lambda _: launch(check=True))
stop_button.on_click(stop)
if busy():
    run_button.disabled = True
    check_button.disabled = True
    status.value = "<b>기존 시연이 실행 중입니다.</b> · 중지하거나 해당 창을 닫으세요."

panel = widgets.VBox([
    widgets.HTML(f"<h3>Classroom Vision</h3><p>{html.escape(platform.system())} · 자동 장치: <b>{automatic.upper()}</b></p>"),
    source, camera, video, device, size, fullscreen,
    widgets.HBox([run_button, stop_button, check_button]), status, output,
])
display(panel)